In [1]:
# IMPORT LIBRARY
import os
import random
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# MEMUAT DATASET
data = np.load("lstm_dataset_split_9.npz")

X_train = data["X_train"]
Y_train = data["Y_train"]
X_val   = data["X_val"]
Y_val   = data["Y_val"]

n_timesteps = X_train.shape[1]
n_features  = X_train.shape[2]

In [3]:
# ARSITEKTUR MODEL LSTM
def build_lstm(params, n_timesteps, n_features):

    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))

    for i in range(params["layers"] - 1):
        model.add(LSTM(
            params["units"],
            return_sequences=True,
            kernel_regularizer=l2(params["weight_decay"])
        ))

    model.add(LSTM(
        params["units"],
        kernel_regularizer=l2(params["weight_decay"])
    ))

    model.add(Dense(1))

    optimizer = Adam(learning_rate=params["learning_rate"])

    model.compile(
        optimizer=optimizer,
        loss="mse"
    )

    return model

In [4]:
# TUNING HIPERPARAMETER
file_csv = "hasil_optimasi.csv"

if os.path.exists(file_csv):
    df_results = pd.read_csv(file_csv)
    results = df_results.to_dict('records')
    print(f"Melanjutkan dari {len(results)} percobaan sebelumnya...")
else:
    results = []
    print("Memulai eksperimen baru...")

# Parameter awal
baseline_params = {
    "layers": 2,
    "units": 64,
    "learning_rate": 0.001,
    "minibatch_size": 256,
    "weight_decay": 0.0
}

if len(results) == 0:
    model = build_lstm(baseline_params, n_timesteps, n_features)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=50,
        batch_size=baseline_params["minibatch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    val_loss = min(history.history["val_loss"])

    results.append({**baseline_params, "val_loss": val_loss})
    pd.DataFrame(results).to_csv(file_csv, index=False)

    best_loss = val_loss
    best_params = baseline_params
    best_model = model

    # Menyimpan model terbaik awal
    best_model.save("best_model.h5")

else:
    # Mengambil model terbaik dari file
    best_entry = min(results, key=lambda x: x["val_loss"])
    best_loss = best_entry["val_loss"]
    best_params = best_entry
    best_model = None

# Melakukan pengujian setiap hiperparameter
total_iterasi = 30

for i in range(len(results), total_iterasi):

    params = {
        "layers": random.choice([2, 3, 4]),
        "units": random.choice([32, 64, 128]),
        "learning_rate": random.uniform(0.0005, 0.005),
        "minibatch_size": random.choice([128, 256, 512]),
        "weight_decay": random.uniform(0.0, 1e-4)
    }

    print(f"\nModel {i+1}/{total_iterasi}")
    print(params)

    model = build_lstm(params, n_timesteps, n_features)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=50,
        batch_size=params["minibatch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    val_loss = min(history.history["val_loss"])

    # Menyimpan hasil
    results.append({**params, "val_loss": val_loss})
    pd.DataFrame(results).to_csv(file_csv, index=False)

    # Memperbarui model terbaik
    if val_loss < best_loss:
        best_loss = val_loss
        best_params = params
        best_model = model

        # Menyimpan model terbaik
        best_model.save("best_model.h5")
        print(">> Model terbaik diperbarui dan disimpan!")

print("\nBest validation loss:", best_loss)
print("Best parameters:", best_params)

Memulai eksperimen baru...



Model 2/30
{'layers': 3, 'units': 128, 'learning_rate': 0.0006720142997559171, 'minibatch_size': 256, 'weight_decay': 2.8290891662020593e-06}

Model 3/30
{'layers': 3, 'units': 128, 'learning_rate': 0.0024937777899571158, 'minibatch_size': 256, 'weight_decay': 5.1407071003953454e-05}

Model 4/30
{'layers': 3, 'units': 128, 'learning_rate': 0.004214673917557556, 'minibatch_size': 128, 'weight_decay': 3.4040082035737496e-05}

Model 5/30
{'layers': 4, 'units': 32, 'learning_rate': 0.004135856661913247, 'minibatch_size': 128, 'weight_decay': 9.347170542176188e-05}

Model 6/30
{'layers': 2, 'units': 64, 'learning_rate': 0.0018054631202868044, 'minibatch_size': 512, 'weight_decay': 3.70466872384288e-05}

Model 7/30
{'layers': 4, 'units': 32, 'learning_rate': 0.0025700250550706677, 'minibatch_size': 256, 'weight_decay': 2.972472213563292e-05}

Model 8/30
{'layers': 2, 'units': 64, 'learning_rate': 0.004479039994864532, 'minibatch_size': 256, 'weight_decay': 3.086367258273708e-05}

Model 9/30